# Fase 2 — Transfer Learning mejorado (MobileNetV2 / MobileNetV3Small)

Este notebook está pensado para la **segunda fase del challenge**.

## Objetivos
1. Cargar un dataset real desde carpetas.
2. Entrenar una CNN propia como baseline.
3. Aplicar `transfer learning` con MobileNet.
4. Probar una fase de `fine-tuning`.
5. Comparar resultados y extraer conclusiones.


## 0. Identificación del equipo

Rellenad esto al empezar.


In [ ]:
TEAM_NAME = ""
TEAM_MEMBERS = [
    "",
    "",
]
MODEL_BASELINE = ""


## 1. Imports


In [ ]:
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2, MobileNetV3Small
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as preprocess_v2
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input as preprocess_v3


## 2. Configuración


In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32 # este puede ser modificado dependiendo de la memoria de tu GPU
EPOCHS = 10 # este es un número pequeño para pruebas rápidas, puedes aumentarlo para mejorar el rendimiento
DATA_DIR = Path("dataset_chihuahua_muffin")
USE_MOBILENET = "v2"  # "v2" o "v3"

assert DATA_DIR.exists(), f"No existe {DATA_DIR}"


## 3. Carga del dataset desde carpetas


In [ ]:
# Cargad aquí los conjuntos de train, val y test a partir de las carpetas del dataset.
# Decidid qué opciones queréis fijar en cada uno: tamaño de imagen, batch size, barajado, etc.
# Al final, guardad también los nombres de clase en una variable como `class_names`.


## 4. Visualización rápida


In [ ]:
plt.figure(figsize=(8, 8))
for images, labels in train_ds.take(1):
    for i in range(min(9, len(images))):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i])])
        plt.axis("off")
plt.tight_layout()


## 5. Optimización del pipeline

Usad `cache`, `shuffle` y `prefetch` para acelerar el entrenamiento.


In [ ]:
# Optimizad aquí el pipeline para train, val y test.
# Podéis usar `cache`, `shuffle`, `prefetch` y cualquier otra mejora que consideréis útil.


## 6. Data augmentation

Esto se aplicará **solo en entrenamiento**.


In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="data_augmentation")


## 7. Baseline from scratch

Primero entrenad una CNN propia para compararla después con MobileNet.


In [ ]:
def build_scratch_model():
    model = tf.keras.Sequential([
        layers.Input(shape=(*IMG_SIZE, 3)),
        data_augmentation,
        layers.Rescaling(1./255),
        # Diseñad aquí vuestra CNN baseline.
        # Decidid cuántas capas usar, de qué tipo, y si queréis añadir dropout o regularización.
        ...
    ])
    return model

scratch_model = build_scratch_model()

# Compilad aquí el modelo con las decisiones que consideréis adecuadas
# (optimizador, learning rate, función de pérdida y métricas).
scratch_model.compile(...)
scratch_model.summary()


In [ ]:
# Definid aquí los callbacks que queráis usar.
# Por ejemplo: EarlyStopping, ReduceLROnPlateau o ModelCheckpoint.

start = time.time()
scratch_history = scratch_model.fit(
    train_ds_opt,
    validation_data=val_ds_opt,
    epochs=...,
    callbacks=...,
    verbose=1
)
scratch_time = time.time() - start


In [ ]:
scratch_test_loss, scratch_test_acc = scratch_model.evaluate(test_ds_opt, verbose=0)
print(f"Scratch test acc: {scratch_test_acc:.4f}")


## 8. Funciones auxiliares para gráficas


In [ ]:
def plot_history(history, title="Modelo"):
    plt.figure(figsize=(10, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(history.history["loss"], label="train")
    plt.plot(history.history["val_loss"], label="val")
    plt.title(f"Loss - {title}")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history["accuracy"], label="train")
    plt.plot(history.history["val_accuracy"], label="val")
    plt.title(f"Accuracy - {title}")
    plt.legend()
    plt.tight_layout()
    plt.show()


## 9. Transfer learning — seleccionar MobileNet

Elegid aquí si vais a trabajar con MobileNetV2 o MobileNetV3Small.


In [ ]:
# Elegid aquí qué versión de MobileNet vais a usar.
# Según esa elección, asignad la clase base, la función de preprocesado
# y un nombre descriptivo para mostrar en pantalla.


## 10. Modelo transfer learning (feature extractor congelado)


In [ ]:
def build_transfer_model(base_class, preprocess_fn, trainable_base=False):
    base_model = base_class(
        input_shape=(*IMG_SIZE, 3),
        include_top=False,
        weights="imagenet"
    )
    base_model.trainable = trainable_base

    inputs = layers.Input(shape=(*IMG_SIZE, 3))
    x = data_augmentation(inputs)
    x = preprocess_fn(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    # Diseñad aquí vuestra cabeza de clasificación.
    # Podéis probar con dropout, una o varias capas densas, etc.
    outputs = ...

    model = tf.keras.Model(inputs, outputs)
    return model, base_model

transfer_model, base_model = build_transfer_model(
    BaseClass, preprocess_fn, trainable_base=False
)

# Compilad aquí el modelo con los hiperparámetros que consideréis adecuados.
transfer_model.compile(...)
transfer_model.summary()


In [ ]:
# Entrenad aquí el modelo con la base congelada.
# Decidid cuántas épocas usar y qué callbacks os interesa mantener.

start = time.time()
transfer_history = transfer_model.fit(
    train_ds_opt,
    validation_data=val_ds_opt,
    epochs=...,
    callbacks=...,
    verbose=1
)
transfer_time = time.time() - start


In [ ]:
transfer_test_loss, transfer_test_acc = transfer_model.evaluate(test_ds_opt, verbose=0)
print(f"{chosen_name} congelado - test acc: {transfer_test_acc:.4f}")
plot_history(transfer_history, f"{chosen_name} congelado")


## 11. Fine-tuning parcial

Descongelad solo una parte de las últimas capas y bajad el learning rate.


In [ ]:
# Activad aquí el fine-tuning parcial.
# Decidid cuántas capas queréis dejar entrenables y justificad esa elección.
base_model.trainable = True

# Ejemplo orientativo:
# for layer in base_model.layers[:-N]:
#     layer.trainable = False

# Recompilad con un learning rate más bajo que en la fase anterior.
transfer_model.compile(...)

start = time.time()
finetune_history = transfer_model.fit(
    train_ds_opt,
    validation_data=val_ds_opt,
    epochs=...,
    callbacks=...,
    verbose=1
)
finetune_time = time.time() - start


In [ ]:
finetune_test_loss, finetune_test_acc = transfer_model.evaluate(test_ds_opt, verbose=0)
print(f"{chosen_name} fine-tuned - test acc: {finetune_test_acc:.4f}")
plot_history(finetune_history, f"{chosen_name} fine-tuned")


## 12. Comparación final


In [ ]:
# Mostrad aquí una comparación final clara entre los tres enfoques.
# Podéis hacerlo con `print`, con una tabla, con un diccionario o como prefiráis.
# Lo importante es que se vea bien qué modelo rinde mejor y cuánto tarda cada uno.


## 13. Preguntas obligatorias para el equipo

Responded aquí o en el documento de entrega:

1. ¿Qué modelo generaliza mejor?
2. ¿Cuál entrena más rápido?
3. ¿Qué diferencias habéis observado entre la base congelada y el fine-tuning?
4. ¿Qué solución final elegiríais y por qué?


## 14. Resultado para leaderboard

Rellenad esto al final con vuestro mejor resultado.


In [ ]:
FINAL_MODEL = ""
FINAL_TEST_ACCURACY = 0.0
FINAL_NOTES = ""
print(TEAM_NAME, FINAL_MODEL, FINAL_TEST_ACCURACY, FINAL_NOTES)
